# 🧠 RNN (Recurrent Neural Network) 심화 학습 가이드

---

## 목차
1. [순차 데이터(Sequential Data)의 이해](#1-순차-데이터sequential-data의-이해)
2. [RNN의 필요성과 기본 구조](#2-rnn의-필요성과-기본-구조)
3. [Vanilla RNN 심화](#3-vanilla-rnn-심화)
4. [BPTT (Backpropagation Through Time)](#4-bptt-backpropagation-through-time)
5. [RNN의 한계와 기울기 문제](#5-rnn의-한계와-기울기-문제)
6. [LSTM (Long Short-Term Memory)](#6-lstm-long-short-term-memory)
7. [GRU (Gated Recurrent Unit)](#7-gru-gated-recurrent-unit)
8. [자연어 처리와 임베딩](#8-자연어-처리와-임베딩)
9. [종합 연습 문제](#9-종합-연습-문제)

---

# 1. 순차 데이터(Sequential Data)의 이해

## 1.1 순차 데이터란?

순차 데이터(Sequential Data)는 **순서(Order)를 가지고 있는 연속적인 데이터**입니다. 핵심 특징은 각 시점(Time step)의 데이터가 **이전 시점의 데이터와 독립적이지 않다**는 것입니다.

### 📌 핵심 개념

**시간적 의존성(Temporal Dependency)**:
- 특정 시점 $t$에서의 데이터는 이전 시점들 $(t_0, t_1, ..., t_{n-1})$의 영향을 받음
- 예: 주가 예측에서 1월 3일의 주가는 1월 2일, 1월 1일의 주가에 영향을 받음

### 순차 데이터의 대표적인 예시

| 유형 | 예시 | 설명 |
|------|------|------|
| 시계열 데이터 | 주가, 기온, 심박수 | 시간에 따른 연속적 측정값 |
| 자연어 데이터 | 문장, 대화 | 단어의 순서가 의미를 결정 |
| DNA 염기서열 | ATCG... | 위치에 따른 유전정보 |
| 음성 신호 | 오디오 파형 | 시간에 따른 진폭 변화 |

## 1.2 순서의 중요성

자연어에서 순서가 얼마나 중요한지 살펴봅시다:

```
"철수가 영희를 좋아한다" ≠ "영희가 철수를 좋아한다"
```

같은 단어들이지만 순서에 따라 **완전히 다른 의미**를 가집니다!

## 📝 연습문제 1.1

**문제**: 다음 중 순차 데이터의 특성으로 **올바르지 않은** 것은?

A) 각 시점의 데이터는 이전 시점과 독립적이다  
B) 데이터의 순서가 중요한 의미를 가진다  
C) 시간적 의존성(Temporal Dependency)을 가진다  
D) 자연어, 시계열 데이터가 대표적인 예시이다  

<details>
<summary>🔍 정답 확인</summary>

**정답: A**

순차 데이터의 핵심 특성은 각 시점의 데이터가 이전 시점과 **독립적이지 않다**는 것입니다. 즉, 서로 의존성을 가집니다. 이것이 RNN이 필요한 이유이기도 합니다.
</details>

## 📝 연습문제 1.2

**문제**: 주가 예측 모델을 구축한다고 가정합니다. 다음 설명 중 올바른 것을 모두 고르세요.

① 3월 15일의 주가를 예측할 때, 3월 14일의 주가만 고려하면 충분하다  
② 과거의 주가 패턴이 미래 예측에 도움이 될 수 있다  
③ 주가 데이터에서 날짜 순서는 중요하지 않다  
④ 주가 예측은 전형적인 순차 데이터 처리 문제이다  

<details>
<summary>🔍 정답 확인</summary>

**정답: ②, ④**

- ① 틀림: 더 긴 기간의 패턴을 고려하면 더 좋은 예측이 가능합니다 (장기 의존성)
- ② 맞음: 과거 패턴 분석은 시계열 예측의 핵심입니다
- ③ 틀림: 시계열 데이터에서 순서는 매우 중요합니다
- ④ 맞음: 주가 예측은 대표적인 시계열/순차 데이터 문제입니다
</details>

---

# 2. RNN의 필요성과 기본 구조

## 2.1 왜 기존 신경망으로는 부족한가?

### MLP(Multi-Layer Perceptron)의 한계

MLP는 고정된 크기의 입력을 받아 고정된 크기의 출력을 생성합니다.

```
문제점:
- 가변 길이 입력 처리 불가
- 순서 정보 반영 불가
- 시간적 의존성 학습 불가
```

### CNN(Convolutional Neural Network)의 한계

CNN은 지역적 패턴을 잘 잡아내지만:
- 고정된 receptive field로 인해 **장거리 의존성** 학습이 어려움
- 순서 정보보다는 공간적 패턴에 특화

## 2.2 RNN의 핵심 아이디어

**"단어(입력 데이터)를 순차적으로 모델에 넣는다!"**

$$\text{RNN의 입력} = (\text{현재 입력}) + (\text{이전 출력/상태})$$

예시:
```
f(f(f(영희), 철수), 좋아해서) → 고백한다
```

## 2.3 RNN Cell의 구조

RNN Cell은 **순환 신경망의 기본 단위**입니다.

### 수식으로 이해하기

$$h_t = \tanh(W_{hh} \cdot h_{t-1} + W_{xh} \cdot x_t + b_h)$$
$$y_t = W_{hy} \cdot h_t + b_y$$

**변수 설명:**
- $x_t$: 시점 $t$의 입력
- $h_{t-1}$: 이전 시점의 은닉 상태 (hidden state)
- $h_t$: 현재 시점의 은닉 상태
- $y_t$: 현재 시점의 출력
- $W_{xh}$: 입력 → 은닉층 가중치
- $W_{hh}$: 은닉층 → 은닉층 가중치 (순환 연결)
- $W_{hy}$: 은닉층 → 출력 가중치

## 📝 연습문제 2.1

**문제**: RNN Cell에서 "순환(Recurrent)"이 의미하는 것으로 가장 적절한 것은?

A) 입력 데이터가 반복적으로 사용된다  
B) 이전 시점의 출력(은닉 상태)이 다음 시점의 입력으로 사용된다  
C) 가중치가 매 시점마다 새로 초기화된다  
D) 역전파가 순환적으로 진행된다  

<details>
<summary>🔍 정답 확인</summary>

**정답: B**

RNN의 "순환"은 **이전 시점의 은닉 상태($h_{t-1}$)가 현재 시점의 계산에 입력으로 사용**되는 것을 의미합니다. 이를 통해 과거 정보를 현재 계산에 반영할 수 있습니다.
</details>

## 📝 연습문제 2.2

**문제**: 다음 RNN 수식에서 각 가중치 행렬의 역할을 연결하세요.

$$h_t = \tanh(W_{hh} \cdot h_{t-1} + W_{xh} \cdot x_t + b_h)$$

| 가중치 | 역할 |
|--------|------|
| 1) $W_{xh}$ | a) 이전 은닉 상태를 변환 |
| 2) $W_{hh}$ | b) 현재 입력을 변환 |
| 3) $b_h$ | c) 편향(bias) 값 |

<details>
<summary>🔍 정답 확인</summary>

**정답:**
- 1) $W_{xh}$ → b) 현재 입력을 변환 (Input-to-Hidden)
- 2) $W_{hh}$ → a) 이전 은닉 상태를 변환 (Hidden-to-Hidden)
- 3) $b_h$ → c) 편향(bias) 값

$W_{xh}$는 입력 $x_t$에 곱해지고, $W_{hh}$는 이전 은닉 상태 $h_{t-1}$에 곱해집니다.
</details>

---

# 3. Vanilla RNN 심화

## 3.1 Vanilla RNN의 구조

가장 기본적인 형태의 RNN을 **Vanilla RNN**이라고 합니다.

### 핵심 특징

1. **동일한 구조의 반복**: 네트워크 A가 병렬적으로 연결
2. **가중치 공유(Parameter Sharing)**: 모든 시점에서 동일한 가중치 사용
3. **tanh 활성화 함수 사용**: 출력 범위 [-1, 1]

### 수식

$$h_t = \tanh(W_{hh} \cdot h_{t-1} + W_{xh} \cdot x_t + b_h)$$
$$y_t = W_{hy} \cdot h_t + b_y$$

## 3.2 Parameter Sharing의 중요성

```
✅ 장점:
- 가변 길이 시퀀스 처리 가능
- 파라미터 수 감소 (메모리 효율적)
- 일반화 능력 향상

⚠️ 주의:
- 같은 층(layer) 내에서만 가중치 공유
- 다른 층의 가중치는 서로 다름
```

## 3.3 Hidden State의 의미

Hidden State $h_t$는 **시점 $t$까지의 모든 입력 정보를 압축**한 벡터입니다.

- 상관관계, 경향성 정보 저장
- 일종의 **메모리(Memory)** 역할
- 시퀀스가 길어질수록 초기 정보는 희석됨 (한계점)

## 3.4 RNN의 종류 (입출력 구조별)

| 구조 | 입력 | 출력 | 응용 예시 |
|------|------|------|----------|
| One-to-One | 하나 | 하나 | 일반 분류 |
| One-to-Many | 하나 | 여러 개 | Image Captioning |
| Many-to-One | 여러 개 | 하나 | 감성 분석 |
| Many-to-Many (동기) | 여러 개 | 여러 개 | 품사 태깅 |
| Many-to-Many (비동기) | 여러 개 | 여러 개 | 기계 번역 |

## 📝 연습문제 3.1

**문제**: Vanilla RNN에서 Parameter Sharing(가중치 공유)에 대한 설명으로 **틀린** 것은?

A) 모든 시점에서 같은 가중치($W_{xh}, W_{hh}, W_{hy}$)를 사용한다  
B) 가변 길이 시퀀스를 처리할 수 있게 해준다  
C) 층(Layer)이 여러 개인 경우에도 모든 층이 같은 가중치를 공유한다  
D) 파라미터 수를 줄여 메모리 효율성을 높인다  

<details>
<summary>🔍 정답 확인</summary>

**정답: C**

가중치 공유는 **같은 층 내에서만** 적용됩니다. 즉, 옆으로(시간 방향) 펼쳐진 같은 층의 셀들은 동일한 가중치를 사용하지만, 위로 쌓인 다른 층들은 **각각 다른 가중치**를 가집니다.
</details>

## 📝 연습문제 3.2

**문제**: 다음 응용 사례와 적합한 RNN 구조를 연결하세요.

| 사례 | 구조 |
|------|------|
| 1) 영화 리뷰 → 긍정/부정 분류 | a) One-to-Many |
| 2) 이미지 → 설명 문장 생성 | b) Many-to-One |
| 3) 한국어 문장 → 영어 문장 번역 | c) Many-to-Many |

<details>
<summary>🔍 정답 확인</summary>

**정답:**
- 1) 영화 리뷰 → 긍정/부정 → **b) Many-to-One** (여러 단어 → 하나의 분류)
- 2) 이미지 → 설명 문장 → **a) One-to-Many** (하나의 이미지 → 여러 단어)
- 3) 한국어 → 영어 → **c) Many-to-Many** (여러 단어 → 여러 단어)
</details>

## 📝 연습문제 3.3

**문제**: 다음 조건의 Vanilla RNN에서 필요한 파라미터 개수를 계산하세요.

```
조건:
- Input dimension: 10
- Hidden dimension: 20
- Output dimension: 5
- Sequence length: 100
```

계산해야 할 것:
1. $W_{xh}$의 파라미터 수
2. $W_{hh}$의 파라미터 수
3. $W_{hy}$의 파라미터 수
4. 총 파라미터 수 (bias 제외)

<details>
<summary>🔍 정답 확인</summary>

**정답:**

1. $W_{xh}$: Input(10) → Hidden(20) = **10 × 20 = 200개**
2. $W_{hh}$: Hidden(20) → Hidden(20) = **20 × 20 = 400개**
3. $W_{hy}$: Hidden(20) → Output(5) = **20 × 5 = 100개**
4. **총 파라미터 수: 200 + 400 + 100 = 700개**

⚠️ **중요**: Sequence length(100)는 파라미터 수에 영향을 주지 않습니다! 이것이 Parameter Sharing의 핵심입니다. 시퀀스가 1000개가 되어도 파라미터 수는 동일합니다.
</details>

---

# 4. BPTT (Backpropagation Through Time)

## 4.1 BPTT란?

BPTT는 RNN에서 사용하는 역전파 알고리즘입니다. 일반 역전파와 달리 **시간 축을 따라** 기울기를 전파합니다.

### 왜 특별한 방법이 필요한가?

RNN에서는 모든 이전 은닉 상태가 다음 상태에 영향을 주므로:
- 기울기 계산 시 **시간을 거슬러 올라가며** 누적
- 체인 룰(Chain Rule)이 시간 방향으로 확장

## 4.2 손실 함수

전체 손실은 각 시점의 손실을 모두 합산:

$$L = \sum_{t=1}^{T} \ell(\hat{y}_t, y_t)$$

## 4.3 BPTT 수식 유도

### 순전파 (Forward Pass)

$$h_t = \sigma(W_h h_{t-1} + W_x x_t + b_h)$$
$$\hat{y}_t = f(h_t) = W_y h_t + b_y$$

### 역전파 (Backward Pass)

$W_h$에 대한 기울기를 구해봅시다:

$$\frac{\partial L}{\partial W_h} = \sum_{t=1}^{T} \left( \frac{\partial \ell_t}{\partial h_t} \cdot \sum_{k=0}^{t-1} \left( \prod_{j=k}^{t-1} \frac{\partial h_{j+1}}{\partial h_j} \right) \frac{\partial h_k}{\partial W_h} \right)$$

### 핵심 포인트: $\frac{\partial h_t}{\partial h_{t-1}}$

이 항이 **연속적으로 곱해지면서** 기울기 소실/폭발 문제가 발생합니다!

$$\frac{\partial h_t}{\partial h_{t-1}} = W_h^T \cdot \text{diag}(\sigma'(z_{t-1}))$$

- $\sigma' = \tanh' = 1 - \tanh^2(x)$: 최댓값이 1, 대부분 1보다 작음
- 이 값들이 여러 번 곱해지면 → **0에 수렴** (기울기 소실)

## 📝 연습문제 4.1

**문제**: BPTT에서 기울기가 "시간을 거슬러" 전파된다는 것의 의미로 가장 적절한 것은?

A) 미래 시점의 데이터를 사용하여 현재를 예측한다  
B) 현재 시점의 손실이 과거 시점의 파라미터에도 영향을 준다  
C) 학습 데이터를 역순으로 처리한다  
D) 시간 순서와 관계없이 병렬로 계산한다  

<details>
<summary>🔍 정답 확인</summary>

**정답: B**

BPTT에서 "시간을 거슬러"란, 시점 $t$에서 계산된 손실이 그 시점뿐만 아니라 **이전 시점들($t-1, t-2, ...$)의 파라미터 업데이트에도 영향**을 미친다는 것입니다. 이는 $h_t$가 $h_{t-1}$에 의존하고, $h_{t-1}$은 $h_{t-2}$에 의존하는 연쇄적 관계 때문입니다.
</details>

## 📝 연습문제 4.2

**문제**: tanh 함수의 미분값 $\tanh'(x) = 1 - \tanh^2(x)$의 특성과 BPTT에서의 영향을 설명하세요.

1) tanh'(x)의 최댓값은 얼마인가?
2) tanh'(x)가 가장 클 때의 x 값은?
3) 이 특성이 긴 시퀀스에서 어떤 문제를 일으키는가?

<details>
<summary>🔍 정답 확인</summary>

**정답:**

1) **최댓값: 1** (tanh(0) = 0일 때, 1 - 0² = 1)

2) **x = 0일 때** tanh'(x) = 1로 최대

3) **기울기 소실 문제(Vanishing Gradient)**:
   - tanh'(x)는 항상 0 ≤ tanh'(x) ≤ 1
   - |x|가 커지면 tanh'(x) → 0에 수렴
   - 긴 시퀀스에서 이 값들이 연속적으로 곱해지면:
     $$\prod_{j=k}^{t-1} \tanh'(z_j) \approx 0$$
   - 초기 시점의 기울기가 거의 0이 되어 학습 불가
</details>

---

# 5. RNN의 한계와 기울기 문제

## 5.1 기울기 소실 (Vanishing Gradient)

### 원인

BPTT에서 기울기는 다음과 같이 계산됩니다:

$$\frac{\partial h_t}{\partial h_k} = \prod_{j=k}^{t-1} \frac{\partial h_{j+1}}{\partial h_j}$$

각 항이 1보다 작으면, 시퀀스가 길어질수록 기울기가 **기하급수적으로 감소**합니다.

### 결과

```
예시: "나는 한국에서 태어났고 ... (100개의 단어) ... 그래서 한국어를 잘한다"

문제: "한국어를 잘한다"를 예측할 때 "한국에서 태어났고"의 정보가
      거의 전달되지 않음 → 장기 의존성 학습 실패
```

## 5.2 기울기 폭발 (Exploding Gradient)

각 항이 1보다 크면, 기울기가 **기하급수적으로 증가**합니다.

### 해결책: Gradient Clipping

$$g \leftarrow \min\left(1, \frac{\theta}{\|g\|}\right) \cdot g$$

기울기의 norm이 임계값 $\theta$를 넘으면 잘라냅니다.

## 5.3 장기 의존성 문제 (Long-term Dependency)

**문제**: 먼 과거의 정보가 현재에 영향을 미쳐야 할 때 학습이 어려움

**해결책**: LSTM, GRU 등 게이트 메커니즘 도입

## 📝 연습문제 5.1

**문제**: 다음 중 Vanilla RNN에서 기울기 소실 문제가 발생하는 근본적인 원인은?

A) 학습률(learning rate)이 너무 작기 때문  
B) tanh 함수의 미분값이 항상 1 이하이기 때문  
C) 파라미터 수가 너무 많기 때문  
D) 배치 크기가 작기 때문  

<details>
<summary>🔍 정답 확인</summary>

**정답: B**

tanh 함수의 미분값 $1 - \tanh^2(x)$는 **최댓값이 1이고 대부분 1보다 작습니다**. BPTT에서 이 값들이 시간 단계마다 곱해지면서 $0.9^{100} \approx 0.00003$처럼 기하급수적으로 감소합니다. 이것이 기울기 소실의 근본 원인입니다.
</details>

## 📝 연습문제 5.2

**문제**: 시퀀스 길이가 50인 RNN에서 BPTT를 수행합니다. 각 시점에서 $\frac{\partial h_{j+1}}{\partial h_j}$의 평균값이 0.9라고 가정할 때, 시점 50에서 시점 1까지 전파되는 기울기는 원래의 약 몇 %가 될까요?

(힌트: $0.9^{49}$을 계산하세요)

<details>
<summary>🔍 정답 확인</summary>

**정답: 약 0.5% (정확히는 약 0.52%)**

계산:
$$0.9^{49} \approx 0.00515$$

즉, 원래 기울기의 **약 0.5%만** 시점 1에 전달됩니다. 이것이 장기 의존성을 학습하기 어려운 이유입니다. 시퀀스가 100이라면 $0.9^{99} \approx 0.00003$으로 거의 0에 가깝습니다.
</details>

---

# 6. LSTM (Long Short-Term Memory)

## 6.1 LSTM이란?

LSTM은 **기울기 소실 문제를 해결**하기 위해 1997년 Hochreiter & Schmidhuber가 제안한 구조입니다.

### 핵심 아이디어

**기억 소자를 두 개로 분리!**
- **Cell State ($C_t$)**: 장기 기억 저장소
- **Hidden State ($h_t$)**: 단기 기억 / 출력

## 6.2 LSTM의 구성 요소

### 3개의 게이트 + 1개의 후보

| 게이트 | 수식 | 역할 |
|--------|------|------|
| Forget Gate ($f_t$) | $\sigma(W_f[h_{t-1}, x_t] + b_f)$ | 기존 정보 중 버릴 것 결정 |
| Input Gate ($i_t$) | $\sigma(W_i[h_{t-1}, x_t] + b_i)$ | 새 정보 중 저장할 것 결정 |
| Cell 후보 ($\tilde{C}_t$) | $\tanh(W_C[h_{t-1}, x_t] + b_C)$ | 저장할 새로운 정보 후보 |
| Output Gate ($o_t$) | $\sigma(W_o[h_{t-1}, x_t] + b_o)$ | 출력할 정보 결정 |

### Cell State 업데이트

$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

- $f_t \odot C_{t-1}$: 기존 정보 중 유지할 부분
- $i_t \odot \tilde{C}_t$: 새로 추가할 정보

### Hidden State 계산

$$h_t = o_t \odot \tanh(C_t)$$

## 6.3 LSTM이 기울기 소실을 해결하는 방법

### Cell State의 "고속도로" 구조

```
C_{t-1} ────────────────────────> C_t
          ↑ (곱셈: 일부 삭제)  ↑ (덧셈: 새 정보 추가)
```

**핵심**:
1. **덧셈 연산**: 곱셈만 있던 Vanilla RNN과 달리 덧셈으로 정보 추가
2. **Forget Gate**: 필요한 정보만 선택적으로 유지
3. **기울기 흐름**: Cell State를 통해 기울기가 비교적 안정적으로 전파

## 📝 연습문제 6.1

**문제**: LSTM의 각 게이트 역할을 올바르게 설명한 것을 모두 고르세요.

① Forget Gate: 이전 Cell State에서 어떤 정보를 버릴지 결정  
② Input Gate: 현재 입력에서 어떤 정보를 Cell State에 추가할지 결정  
③ Output Gate: 다음 시점에 전달할 Cell State를 결정  
④ 모든 게이트는 sigmoid 함수를 사용하여 0~1 사이 값 출력  

<details>
<summary>🔍 정답 확인</summary>

**정답: ①, ②, ④**

- ① 맞음: Forget Gate는 $C_{t-1}$에서 버릴 정보를 결정 (값이 0에 가까우면 삭제)
- ② 맞음: Input Gate는 새 정보($\tilde{C}_t$)의 반영 비율을 결정
- ③ 틀림: Output Gate는 **Hidden State 출력**을 결정합니다. Cell State 자체는 Forget Gate와 Input Gate에 의해 업데이트됩니다.
- ④ 맞음: 모든 게이트는 sigmoid를 사용하여 0~1 범위의 "비율"을 출력
</details>

## 📝 연습문제 6.2

**문제**: LSTM의 Cell State 업데이트 수식 $C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$에서, 다음 상황에서의 $C_t$ 값을 계산하세요.

```
조건:
- C_{t-1} = [1.0, 2.0, 3.0]
- f_t = [0.8, 0.5, 0.2]  (Forget Gate 출력)
- i_t = [0.3, 0.7, 0.9]  (Input Gate 출력)
- \tilde{C}_t = [0.5, 0.5, 0.5]  (Cell 후보)

⊙는 element-wise 곱셈입니다.
```

<details>
<summary>🔍 정답 확인</summary>

**정답: $C_t = [0.95, 1.35, 1.05]$**

계산 과정:

1. $f_t \odot C_{t-1}$:
   - [0.8×1.0, 0.5×2.0, 0.2×3.0] = **[0.8, 1.0, 0.6]**

2. $i_t \odot \tilde{C}_t$:
   - [0.3×0.5, 0.7×0.5, 0.9×0.5] = **[0.15, 0.35, 0.45]**

3. $C_t = [0.8, 1.0, 0.6] + [0.15, 0.35, 0.45]$:
   - = **[0.95, 1.35, 1.05]**

해석:
- 첫 번째 요소: 80% 유지 + 새 정보 0.15 = 0.95
- 두 번째 요소: 50% 유지 + 새 정보 0.35 = 1.35
- 세 번째 요소: 20% 유지 + 새 정보 0.45 = 1.05 (기존 정보 대부분 삭제)
</details>

## 📝 연습문제 6.3

**문제**: LSTM이 Vanilla RNN의 기울기 소실 문제를 완화하는 핵심 메커니즘을 설명하세요.

힌트: Cell State 업데이트 수식의 **연산 종류**에 주목하세요.

<details>
<summary>🔍 정답 확인</summary>

**정답:**

LSTM이 기울기 소실을 완화하는 핵심 메커니즘:

**1. 덧셈 연산의 도입**
- Vanilla RNN: $h_t = \tanh(W \cdot h_{t-1} + ...)$ → **곱셈만** 존재
- LSTM: $C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$ → **덧셈** 포함
- 덧셈의 기울기는 **1**이므로, 기울기가 그대로 전파됨

**2. Cell State "고속도로"**
- Cell State는 시퀀스 전체를 관통하는 직선 경로 형성
- 기울기가 이 경로를 통해 직접 전파 가능

**3. Forget Gate의 역할**
- $f_t$가 1에 가까우면 기울기가 거의 그대로 전파
- 네트워크가 학습을 통해 중요한 정보의 $f_t$를 높게 유지

**수식으로 보면:**
$$\frac{\partial C_t}{\partial C_{t-1}} = f_t$$

Forget Gate가 1에 가까우면 기울기가 거의 손실 없이 전파됩니다.
</details>

---

# 7. GRU (Gated Recurrent Unit)

## 7.1 GRU란?

GRU는 2014년 Cho et al.이 제안한 LSTM의 **간소화 버전**입니다.

### LSTM과의 비교

| 특성 | LSTM | GRU |
|------|------|-----|
| 게이트 수 | 3개 (Forget, Input, Output) | 2개 (Reset, Update) |
| 상태 변수 | 2개 ($C_t$, $h_t$) | 1개 ($h_t$) |
| 파라미터 수 | 더 많음 | 더 적음 |
| 학습 속도 | 상대적으로 느림 | 상대적으로 빠름 |

## 7.2 GRU의 구성 요소

### Reset Gate ($r_t$)
$$r_t = \sigma(W_r \cdot [h_{t-1}, x_t])$$
- 이전 hidden state를 **얼마나 초기화(리셋)할지** 결정
- 값이 0에 가까우면 이전 정보 무시

### Update Gate ($z_t$)
$$z_t = \sigma(W_z \cdot [h_{t-1}, x_t])$$
- **과거 정보와 새 정보의 비율** 결정
- LSTM의 Forget Gate + Input Gate 역할을 하나로 통합

### 후보 Hidden State ($\tilde{h}_t$)
$$\tilde{h}_t = \tanh(W \cdot [r_t \odot h_{t-1}, x_t])$$

### 최종 Hidden State ($h_t$)
$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

## 7.3 GRU의 직관적 이해

```
Update Gate z_t가 결정하는 것:
- z_t → 1: 새 정보(\tilde{h}_t)를 많이 반영
- z_t → 0: 기존 정보(h_{t-1})를 많이 유지

이것이 LSTM의 Forget + Input을 하나로 합친 것!
```

## 📝 연습문제 7.1

**문제**: GRU의 Update Gate가 LSTM의 어떤 게이트들의 역할을 통합했는지 설명하고, 그 수식적 관계를 보이세요.

<details>
<summary>🔍 정답 확인</summary>

**정답:**

GRU의 Update Gate($z_t$)는 LSTM의 **Forget Gate($f_t$)와 Input Gate($i_t$)를 통합**합니다.

**수식적 관계:**

LSTM:
$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

GRU:
$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

**비교:**
- LSTM: $f_t$와 $i_t$가 **독립적**으로 결정
- GRU: $(1-z_t)$와 $z_t$가 **상보적** 관계 (합이 항상 1)

즉, GRU에서는:
- $1 - z_t$ ≈ Forget Gate (기존 정보 유지 비율)
- $z_t$ ≈ Input Gate (새 정보 반영 비율)

**제약 조건 추가**: 두 비율의 합이 항상 1이므로 파라미터가 줄고 학습이 안정화됩니다.
</details>

## 📝 연습문제 7.2

**문제**: 다음 조건에서 GRU의 출력 $h_t$를 계산하세요.

```
조건:
- h_{t-1} = [0.5, 0.8]
- z_t = [0.3, 0.7]  (Update Gate)
- \tilde{h}_t = [0.9, 0.2]  (후보 Hidden State)

수식: h_t = (1 - z_t) ⊙ h_{t-1} + z_t ⊙ \tilde{h}_t
```

<details>
<summary>🔍 정답 확인</summary>

**정답: $h_t = [0.62, 0.38]$**

계산 과정:

1. $(1 - z_t) = [1-0.3, 1-0.7] = [0.7, 0.3]$

2. $(1 - z_t) \odot h_{t-1}$:
   - [0.7×0.5, 0.3×0.8] = **[0.35, 0.24]**

3. $z_t \odot \tilde{h}_t$:
   - [0.3×0.9, 0.7×0.2] = **[0.27, 0.14]**

4. $h_t = [0.35, 0.24] + [0.27, 0.14]$:
   - = **[0.62, 0.38]**

해석:
- 첫 번째 요소: 70% 기존 + 30% 새 정보
- 두 번째 요소: 30% 기존 + 70% 새 정보
</details>

## 📝 연습문제 7.3

**문제**: LSTM과 GRU 중 어떤 것을 선택해야 할지 결정할 때 고려해야 할 사항들을 3가지 이상 나열하세요.

<details>
<summary>🔍 정답 확인</summary>

**정답:**

**1. 데이터 양**
- 데이터가 **적을 때**: GRU (파라미터가 적어 과적합 위험 감소)
- 데이터가 **많을 때**: LSTM (더 복잡한 패턴 학습 가능)

**2. 계산 자원**
- 제한된 자원: GRU (더 빠른 학습/추론)
- 충분한 자원: LSTM 또는 둘 다 실험

**3. 시퀀스 특성**
- 매우 긴 장기 의존성: LSTM (Cell State 분리가 더 효과적일 수 있음)
- 중간 길이 의존성: GRU로도 충분

**4. 해석 가능성**
- 게이트 분석이 필요한 경우 LSTM이 더 직관적일 수 있음

**5. 기존 연구/사례**
- 해당 도메인에서 더 많이 검증된 구조 선택
- NLP에서는 LSTM이 더 오랜 역사와 많은 사례 보유

**실용적 조언**: 둘 다 실험해보고 validation 성능으로 결정하는 것이 가장 확실합니다.
</details>

---

# 8. 자연어 처리와 임베딩

## 8.1 자연어 처리(NLP)란?

자연어 처리(Natural Language Processing)는 **인간의 언어를 컴퓨터가 이해하고 처리**할 수 있도록 하는 기술입니다.

### NLP의 응용 분야

| 난이도 | 응용 | 예시 |
|--------|------|------|
| 쉬움 | 스펠링 체크, 키워드 검사 | 맞춤법 교정 |
| 중간 | 형태 해석, 구문 분석 | 품사 태깅 |
| 어려움 | 기계 번역, 감정 분석, QA | 챗봇, 번역기 |

## 8.2 단어의 수치화: 왜 필요한가?

```
컴퓨터는 문자열을 직접 처리할 수 없습니다!

"자연어 처리" → ??? → 수학적 연산 → 결과
              ↑
          숫자화 필요!
```

## 8.3 One-Hot Encoding

가장 간단한 방법: 각 단어를 고유한 인덱스로 표현

```
어휘: [나는, 영희를, 좋아한다, 철수가]

나는     = [1, 0, 0, 0]
영희를   = [0, 1, 0, 0]
좋아한다 = [0, 0, 1, 0]
철수가   = [0, 0, 0, 1]
```

### One-Hot Encoding의 문제점

1. **차원의 저주**: 어휘 크기 = 벡터 차원 (수십만 차원 가능)
2. **의미 표현 불가**: 모든 단어 간 거리가 동일
   - "사과"와 "바나나"의 거리 = "사과"와 "컴퓨터"의 거리
3. **희소 행렬(Sparse Matrix)**: 대부분 0, 비효율적

## 8.4 임베딩(Embedding)

**해결책**: 단어를 **저차원 밀집 벡터(Dense Vector)**로 표현

```
One-Hot (희소): [1, 0, 0, 0, ..., 0]  (10000차원)
                    ↓
Embedding (밀집): [0.2, -0.5, 0.8, ..., 0.3]  (300차원)
```

### 임베딩의 장점

1. **차원 축소**: 수만 차원 → 수백 차원
2. **의미적 유사성**: 비슷한 단어는 비슷한 벡터
3. **연산 가능**: King - Man + Woman ≈ Queen

### 대표적인 임베딩 방법

- **Word2Vec**: CBOW, Skip-gram
- **GloVe**: 동시 출현 행렬 기반
- **FastText**: 부분단어(subword) 활용
- **BERT, GPT**: 문맥 기반 동적 임베딩

## 📝 연습문제 8.1

**문제**: 어휘 크기가 50,000개인 자연어 처리 모델에서 One-Hot Encoding과 300차원 임베딩을 사용할 때, 단어 하나를 표현하는 데 필요한 메모리(float32 기준)를 각각 계산하세요.

(float32 = 4 bytes)

<details>
<summary>🔍 정답 확인</summary>

**정답:**

**One-Hot Encoding:**
- 벡터 크기: 50,000
- 메모리: 50,000 × 4 bytes = **200,000 bytes ≈ 195 KB**

**300차원 Embedding:**
- 벡터 크기: 300
- 메모리: 300 × 4 bytes = **1,200 bytes ≈ 1.2 KB**

**비교:**
- One-Hot은 임베딩보다 약 **167배** 더 많은 메모리 사용
- 문장이 100단어라면: One-Hot 19.5MB vs Embedding 117KB

이것이 실제 NLP 시스템에서 임베딩이 필수적인 이유입니다!
</details>

## 📝 연습문제 8.2

**문제**: Word2Vec의 유명한 예시 "King - Man + Woman ≈ Queen"이 의미하는 바를 설명하고, 이것이 One-Hot Encoding으로는 불가능한 이유를 설명하세요.

<details>
<summary>🔍 정답 확인</summary>

**정답:**

**"King - Man + Woman ≈ Queen"의 의미:**

Word2Vec 임베딩 공간에서:
- King과 Man의 차이 ≈ Queen과 Woman의 차이
- 이 차이 벡터는 "성별 변환"이라는 의미를 담고 있음
- 즉, **단어 간의 의미적 관계가 벡터 연산으로 표현** 가능

**One-Hot Encoding으로 불가능한 이유:**

One-Hot에서:
```
King  = [1, 0, 0, 0, ...]
Man   = [0, 1, 0, 0, ...]
Woman = [0, 0, 1, 0, ...]
Queen = [0, 0, 0, 1, ...]
```

1. **모든 단어 간 거리가 동일**: 유클리드 거리가 모두 $\sqrt{2}$
2. **의미적 관계 없음**: King-Man = [1, -1, 0, 0, ...]는 의미 없는 벡터
3. **덧셈 결과가 어휘에 없음**: 계산 결과가 유효한 One-Hot 벡터가 아님

임베딩은 **의미 공간**에서의 연산을 가능하게 합니다.
</details>

---

# 9. 종합 연습 문제

이제 전체 내용을 종합하는 문제들입니다.

## 📝 종합문제 1: 개념 비교

**문제**: 다음 표를 완성하세요.

| 항목 | Vanilla RNN | LSTM | GRU |
|------|-------------|------|-----|
| 게이트 수 | ? | ? | ? |
| 상태 변수 | ? | ? | ? |
| 장기 의존성 학습 | ? | ? | ? |
| 파라미터 수 (상대적) | ? | ? | ? |

<details>
<summary>🔍 정답 확인</summary>

| 항목 | Vanilla RNN | LSTM | GRU |
|------|-------------|------|-----|
| 게이트 수 | **0개** | **3개** (Forget, Input, Output) | **2개** (Reset, Update) |
| 상태 변수 | **1개** ($h_t$) | **2개** ($C_t$, $h_t$) | **1개** ($h_t$) |
| 장기 의존성 학습 | **어려움** | **효과적** | **효과적** |
| 파라미터 수 (상대적) | **가장 적음** | **가장 많음** | **중간** |

</details>

## 📝 종합문제 2: 수식 이해

**문제**: 다음 수식들이 어떤 모델의 어떤 부분인지 맞추세요.

1) $h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b)$

2) $C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$

3) $h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$

4) $f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$

<details>
<summary>🔍 정답 확인</summary>

**정답:**

1) **Vanilla RNN** - Hidden State 계산
   - tanh 활성화, 단순한 가중합 구조

2) **LSTM** - Cell State 업데이트
   - Forget Gate로 기존 정보 선택 + Input Gate로 새 정보 추가

3) **GRU** - Hidden State 계산
   - $(1-z_t)$와 $z_t$로 기존/새 정보 비율 결정

4) **LSTM** - Forget Gate
   - sigmoid 출력, $[h_{t-1}, x_t]$ 연결 입력

</details>

## 📝 종합문제 3: 시나리오 분석

**문제**: 당신은 주식 가격 예측 모델을 구축하려 합니다. 다음 질문에 답하세요.

1) 이 문제는 어떤 유형의 RNN 구조(One-to-One, Many-to-One 등)가 적합한가요?

2) 30일간의 가격 데이터로 다음 날 가격을 예측한다면, Vanilla RNN과 LSTM 중 어느 것이 더 적합할까요? 그 이유는?

3) 입력으로 5개의 특성(시가, 고가, 저가, 종가, 거래량)을 사용하고 hidden dimension이 64라면, LSTM의 Forget Gate 가중치 $W_f$의 크기는?

<details>
<summary>🔍 정답 확인</summary>

**정답:**

**1) Many-to-One**
- 30일간의 데이터(Many) → 1일 예측값(One)

**2) LSTM이 더 적합**
- 30일은 상대적으로 긴 시퀀스
- 주가는 며칠 전의 패턴이 현재에 영향 (장기 의존성)
- Vanilla RNN은 기울기 소실로 초기 데이터 학습 어려움
- LSTM의 Cell State가 과거 패턴 정보 효과적으로 보존

**3) $W_f$의 크기: (64 + 5) × 64 = 4,416**

계산:
- $f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$
- 입력: $[h_{t-1}, x_t]$의 크기 = 64 + 5 = 69
- 출력: $f_t$의 크기 = 64 (hidden dimension과 동일)
- 따라서 $W_f$: 69 × 64 = **4,416개 파라미터**

</details>

## 📝 종합문제 4: 오류 찾기

**문제**: 다음 설명에서 틀린 부분을 찾아 수정하세요.

```
"LSTM에서 Forget Gate의 출력이 1에 가까우면 이전 Cell State의 정보를 
많이 삭제하고, 0에 가까우면 정보를 유지한다. Output Gate는 다음 시점에 
전달할 Cell State를 결정하며, Cell State 업데이트 시 곱셈 연산만 
사용하기 때문에 기울기 소실 문제가 발생한다."
```

<details>
<summary>🔍 정답 확인</summary>

**틀린 부분과 수정:**

**오류 1**: "1에 가까우면... 삭제하고, 0에 가까우면 유지"
- **수정**: "1에 가까우면 **유지**하고, 0에 가까우면 **삭제**"
- 이유: $C_t = f_t \odot C_{t-1} + ...$에서 $f_t=1$이면 $C_{t-1}$ 그대로 유지

**오류 2**: "Output Gate는 다음 시점에 전달할 Cell State를 결정"
- **수정**: "Output Gate는 **Hidden State 출력**을 결정"
- 이유: Cell State 전달은 Forget Gate와 Input Gate가 담당

**오류 3**: "곱셈 연산만 사용하기 때문에 기울기 소실"
- **수정**: "**덧셈 연산**을 사용하기 때문에 기울기 소실 문제가 **완화**된다"
- 이유: $C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$에서 **덧셈**이 핵심

**올바른 설명**:
"LSTM에서 Forget Gate의 출력이 1에 가까우면 이전 Cell State의 정보를 많이 **유지**하고, 0에 가까우면 정보를 **삭제**한다. Output Gate는 **Hidden State 출력**을 결정하며, Cell State 업데이트 시 **덧셈 연산**을 사용하기 때문에 기울기 소실 문제가 **완화**된다."
</details>

## 📝 종합문제 5: 개념 정리

**문제**: 다음 빈칸을 채우세요.

1) RNN에서 Parameter Sharing이란 ____________ 에서 ____________ 를 공유하는 것을 의미한다.

2) BPTT에서 기울기 소실이 발생하는 이유는 ____________ 함수의 미분값이 ____________ 이하이기 때문이다.

3) LSTM에서 장기 기억을 담당하는 것은 ____________ 이고, 단기 기억/출력을 담당하는 것은 ____________ 이다.

4) GRU의 Update Gate는 LSTM의 ____________ 와 ____________ 의 역할을 하나로 통합한 것이다.

5) 단어를 저차원 밀집 벡터로 변환하는 것을 ____________ 이라 하며, 대표적인 방법으로 ____________ 가 있다.

<details>
<summary>🔍 정답 확인</summary>

**정답:**

1) **같은 층(layer)** 에서 **가중치(W)** 를 공유

2) **tanh(또는 활성화)** 함수의 미분값이 **1** 이하이기 때문이다.

3) **Cell State ($C_t$)** 이고, **Hidden State ($h_t$)** 이다.

4) **Forget Gate** 와 **Input Gate** 의 역할을 하나로 통합

5) **임베딩(Embedding)** 이라 하며, **Word2Vec (또는 GloVe, FastText 등)** 가 있다.

</details>

---

# 🎯 학습 정리

## 핵심 개념 요약

### 1. 순차 데이터
- 순서가 있고, 시간적 의존성을 가진 데이터
- 예: 자연어, 시계열, 음성

### 2. Vanilla RNN
- 이전 hidden state + 현재 입력 → 현재 hidden state
- Parameter Sharing으로 가변 길이 처리
- 문제: 기울기 소실로 장기 의존성 학습 어려움

### 3. BPTT
- RNN의 역전파 알고리즘
- 시간을 거슬러 기울기 전파
- tanh 미분값의 연속 곱셈으로 기울기 소실

### 4. LSTM
- Cell State(장기) + Hidden State(단기) 분리
- 3개의 게이트: Forget, Input, Output
- 덧셈 연산으로 기울기 소실 완화

### 5. GRU
- LSTM의 간소화 버전
- 2개의 게이트: Reset, Update
- Update Gate = Forget + Input 통합

### 6. 임베딩
- One-Hot → Dense Vector 변환
- 차원 축소 + 의미적 유사성 보존

---

**수고하셨습니다! 🎉**